# Apache Iceberg com PySpark

Notebook de referência para configurar Spark + Iceberg e executar operações `INSERT`, `UPDATE` e `DELETE` no mesmo cenário de dados do projeto.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession

raiz_projeto = Path("..").resolve()
warehouse_path = str(raiz_projeto / "data" / "iceberg_warehouse")

ICEBERG_RUNTIME = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"

spark = (
    SparkSession.builder
    .appName("Iceberg_Statcast")
    .config("spark.jars.packages", ICEBERG_RUNTIME)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.ice", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.ice.type", "hadoop")
    .config("spark.sql.catalog.ice.warehouse", warehouse_path)
    .config("spark.sql.defaultCatalog", "ice")
    .getOrCreate()
)

print("Warehouse Iceberg:", warehouse_path)
spark.sql("SELECT current_catalog() AS catalogo, current_database() AS banco").show(truncate=False)

26/04/23 22:23:58 WARN Utils: Your hostname, DESKTOP-G0CPIMV resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/23 22:23:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ana/DEV/trabalho1ed/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ana/.ivy2/cache
The jars for the packages stored in: /home/ana/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-80d54692-7fe2-4ae0-b229-3bf6a5a04637;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
:: resolution report :: resolve 373ms :: artifacts dl 8ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#sp

Warehouse Iceberg: /home/ana/DEV/trabalho1ed/data/iceberg_warehouse
+--------+-----+
|catalogo|banco|
+--------+-----+
|ice     |     |
+--------+-----+



## Carga inicial da tabela Iceberg

In [2]:
import sys

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.ingestao import ler_e_limpar_dados

CAMINHO_CSV = str(raiz_projeto / "data" / "raw" / "statcast_data.csv")
TABELA = "ice.baseball.statcast_arremessadores"

spark.sql("CREATE NAMESPACE IF NOT EXISTS ice.baseball")
spark.sql(f"DROP TABLE IF EXISTS {TABELA}")

df = ler_e_limpar_dados(spark, CAMINHO_CSV)
df.createOrReplaceTempView("staging_statcast")

spark.sql(
    f"""
    CREATE TABLE {TABELA} (
      nome_jogador STRING,
      total_arremessos INT,
      velocidade_media FLOAT,
      taxa_giro INT,
      media_rebatidas_contra FLOAT,
      strikeouts INT,
      home_runs_cedidos INT,
      walks_cedidos INT
    ) USING iceberg
    """
)

spark.sql(
    f"""
    INSERT INTO {TABELA}
    SELECT * FROM staging_statcast
    """
)

spark.sql(f"SELECT COUNT(*) AS total_linhas FROM {TABELA}").show()

26/04/23 22:24:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+
|total_linhas|
+------------+
|         873|
+------------+



## INSERT no Iceberg

In [3]:
spark.sql(
    f"""
    INSERT INTO {TABELA}
    VALUES (
      'Pitcher, Teste',
      500,
      93.5,
      2400,
      0.210,
      110,
      10,
      30
    )
    """
)

spark.sql(
    f"""
    SELECT nome_jogador, velocidade_media, strikeouts
    FROM {TABELA}
    WHERE nome_jogador = 'Pitcher, Teste'
    """
).show(truncate=False)

+--------------+----------------+----------+
|nome_jogador  |velocidade_media|strikeouts|
+--------------+----------------+----------+
|Pitcher, Teste|93.5            |110       |
+--------------+----------------+----------+



## UPDATE no Iceberg

In [4]:
spark.sql(
    f"""
    UPDATE {TABELA}
    SET strikeouts = strikeouts - 1
    WHERE nome_jogador = 'Webb, Logan'
    """
)

spark.sql(
    f"""
    SELECT nome_jogador, strikeouts, velocidade_media
    FROM {TABELA}
    WHERE nome_jogador = 'Webb, Logan'
    """
).show(truncate=False)

+------------+----------+----------------+
|nome_jogador|strikeouts|velocidade_media|
+------------+----------+----------------+
|Webb, Logan |223       |88.9            |
+------------+----------+----------------+



## DELETE no Iceberg

In [5]:
spark.sql(
    f"""
    DELETE FROM {TABELA}
    WHERE nome_jogador = 'Pitcher, Teste'
    """
)

spark.sql(
    f"""
    SELECT nome_jogador
    FROM {TABELA}
    WHERE nome_jogador = 'Pitcher, Teste'
    """
).show(truncate=False)

+------------+
|nome_jogador|
+------------+
+------------+



## Histórico de snapshots (auditoria Iceberg)

In [6]:
spark.sql(f"SELECT * FROM {TABELA}.snapshots ORDER BY committed_at DESC").show(truncate=False)
spark.sql(f"SELECT * FROM {TABELA}.history ORDER BY made_current_at DESC").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                                  |summary                                                                                                                                                     